# Generate Label Studio XML from AN Taxonomy

In [1]:
import pandas as pd
import ast
from xml.sax.saxutils import escape

In [2]:
with open('data/input/label-studio-ui-template.xml', 'r') as f:
    template = f.read().splitlines() 
template

['<View>',
 '  <Style>',
 '    .ls-group { margin-bottom: 20px; padding: 15px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; }',
 '    .ls-header { font-size: 16px; font-weight: bold; margin-bottom: 10px; color: #2c3e50; border-bottom: 1px solid #ddd; padding-bottom: 5px;}',
 '  </Style>',
 '  ',
 '  <View style="padding: 10px; font-size: 16px; line-height: 1.6; border-bottom: 2px solid #ccc; margin-bottom: 20px;">',
 '    <Text name="text" value="$note_content" />',
 '  </View>',
 '',
 '  <PLACEHOLDER>',
 '',
 '  <View className="ls-group">',
 '    <View className="ls-header"><Text name="hdr_entities" value="Entities &amp; Missed PII"/></View>',
 '    <Labels name="entity_labels" toName="text" choice="multiple">',
 '      <Label value="Pronoun" background="#95a5a6" />',
 '      <Label value="Descriptive Role" background="#7f8c8d" />',
 '      <Label value="Pseudonym" background="#8e44ad" hint="Pre-annotated fake name" />',
 '      <Label value="Missed Person Name

In [3]:
taxonomy = pd.read_csv('data/output/taxonomy_v2_autogen.csv')
taxonomy.columns


Index(['top_level_category', 'high_level_category', 'category_description',
       'values_hint', 'high_level_cat_label', 'cat_label', 'regex'],
      dtype='str')

In [4]:
taxonomy.high_level_cat_label.unique()

<ArrowStringArray>
[           'cat_disability_disability',
            'cat_disability_adaptation',
          'cat_safety_risk_safety_risk',
          'cat_vulnerability_addiction',
               'cat_vulnerability_care',
             'cat_vulnerability_health',
 'cat_vulnerability_housing_conditions',
        'cat_vulnerability_life_events',
          'cat_vulnerability_financial',
           'cat_vulnerability_mobility',
 'cat_service_need_communication_needs']
Length: 11, dtype: str

In [5]:
# Colour per high_level_cat_label
GROUP_COLOURS = {
    'cat_disability_disability':            '#27ae60',
    'cat_disability_adaptation':            '#2980b9',
    'cat_safety_risk_safety_risk':          '#e74c3c',
    'cat_vulnerability_addiction':          '#c0392b',
    'cat_vulnerability_care':               '#16a085',
    'cat_service_need_comms':               '#f1c40f',
    'cat_vulnerability_financial':          '#f39c12',
    'cat_vulnerability_health':             '#2ecc71',
    'cat_vulnerability_housing_conditions': '#3498db',
    'cat_vulnerability_life_events':        '#d35400',
    'cat_vulnerability_mobility':           '#1e8449',
}
DEFAULT_COLOUR = '#95a5a6'

# Group headers per top_level_category
GROUP_HEADERS = {
    'Disability':     ('Disability',        'hdr_disability'),
    'Vulnerability':  ('Vulnerability',     'hdr_vulnerability'),
    'Safety & Risk':  ('Safety &amp; Risk', 'hdr_safety'),
    'Service Needs':  ('Service Needs',      'hdr_service_needs'),
}


# Generate labels selection grouped by top_level_category
lines = []
for top_level, group in taxonomy.groupby('top_level_category', sort=False):
    header_text, header_name = GROUP_HEADERS.get(str(top_level), (escape(top_level), f'hdr_{top_level.lower().replace(" ", "_")}'))
    lines += [
        '',
        '  <View className="ls-group">',
        f'    <View className="ls-header"><Text name="{header_name}" value="{header_text}"/></View>',
    ]

    # Generate labels selection grouped by high_level_category
    for hl_label, hl_group in group.groupby('high_level_cat_label', sort=False):
        colour = GROUP_COLOURS.get(str(hl_label), DEFAULT_COLOUR)
        labels_name = f'labels_{str(hl_label).split("_")[-1]}'
        lines.append(f'    <Labels name="{labels_name}" toName="text" choice="multiple">')
        for row in hl_group.itertuples():
            hint = ", ".join(ast.literal_eval(str(row.values_hint)))
            lines.append(f'      <Label value="{row.cat_label}" html="{escape(str(row.category_description))}" background="{colour}" hint="{hint}"/>')
        lines.append('    </Labels>')
    lines.append('  </View>')

lines


['',
 '  <View className="ls-group">',
 '    <View className="ls-header"><Text name="hdr_disability" value="Disability"/></View>',
 '    <Labels name="labels_disability" toName="text" choice="multiple">',
 '      <Label value="cat_disability_disability_sensory" html="Sensory" background="#27ae60" hint="Deaf, Hearing impairment, Visually impaired, Partially sighted, Deafblindness, Speech impairment"/>',
 '    </Labels>',
 '    <Labels name="labels_adaptation" toName="text" choice="multiple">',
 '      <Label value="cat_disability_adaptation_requires_adapted_property" html="Requires adapted property" background="#2980b9" hint="Minor adaptations, Major adaptation"/>',
 '    </Labels>',
 '  </View>',
 '',
 '  <View className="ls-group">',
 '    <View className="ls-header"><Text name="hdr_safety" value="Safety &amp; Risk"/></View>',
 '    <Labels name="labels_risk" toName="text" choice="multiple">',
 '      <Label value="cat_safety_risk_safety_risk_domestic_abuse" html="Domestic abuse" back

In [6]:
i = template.index("  <PLACEHOLDER>")
new_file = template[:i] + lines + template[i+1:]

with open('data/output/label-studio-ui.xml', 'w') as f:
    f.write('\n'.join(new_file))